# 1. ІМПОРТИ ТА НАЛАШТУВАННЯ

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import difflib
import optuna

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, roc_auc_score

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore') 
pd.set_option('display.max_columns', None) 

print("✅ Бібліотеки Grandmaster-рівня успішно імпортовано!")

# 2. ЗАВАНТАЖЕННЯ ТА ОЧИЩЕННЯ ДАНИХ

In [ ]:
print("📂 Завантажую тренувальні дані...")
users = pd.read_csv('train_users.csv')
transactions = pd.read_csv('train_transactions.csv')

users.drop_duplicates(inplace=True)
transactions.drop_duplicates(inplace=True)
users.fillna('unknown', inplace=True)
transactions.fillna('none', inplace=True)

users["timestamp_reg"] = pd.to_datetime(users["timestamp_reg"], format='mixed', utc=True)
transactions["timestamp_tr"] = pd.to_datetime(transactions["timestamp_tr"], format='mixed', utc=True)

print(f"📊 Тренувальні дані готові: Юзерів - {len(users)}, Транзакцій - {len(transactions)}")

# 3. ФУНКЦІЯ ГЕНЕРАЦІЇ ФІЧЕЙ

In [ ]:
def create_features(df_users, df_trans):
    df_u = df_users.copy()
    df_t = df_trans.copy().sort_values(['id_user', 'timestamp_tr'])
    
    # 1. БАЗОВІ ФІЧІ ТА ПОМИЛКИ
    n_transactions = df_t.groupby('id_user').size().reset_index(name='n_transactions')
    fraud_errors = ['fraud', 'cvv error', 'invalid data', '3ds error', 'do not honor']
    df_t['is_suspicious_error'] = df_t['error_group'].apply(lambda x: 1 if x in fraud_errors else 0)
    suspicious_err = df_t.groupby('id_user')['is_suspicious_error'].sum().reset_index(name='suspicious_errors_count')
    
    # 2. МЕРЕЖА ТА КАРТКИ
    card_users = df_t.groupby('card_mask_hash')['id_user'].nunique().reset_index(name='users_per_card')
    df_t = pd.merge(df_t, card_users, on='card_mask_hash', how='left')
    network_feat = df_t.groupby('id_user')['users_per_card'].max().reset_index(name='max_users_per_card')
    cards_per_user = df_t.groupby('id_user')['card_mask_hash'].nunique().reset_index(name='unique_cards_tried')
    
    # 3. ШВИДКІСТЬ ТА ЧАС
    df_t['time_diff_min'] = df_t.groupby('id_user')['timestamp_tr'].diff().dt.total_seconds() / 60
    velocity_feat = df_t.groupby('id_user').agg(
        avg_time_between_tr=('time_diff_min', 'mean'),
        min_time_between_tr=('time_diff_min', 'min')
    ).reset_index()
    
    df_t_time = pd.merge(df_t[['id_user', 'timestamp_tr', 'amount']], df_u[['id_user', 'timestamp_reg']], on='id_user', how='left')
    df_t_time['hours_since_reg'] = (df_t_time['timestamp_tr'] - df_t_time['timestamp_reg']).dt.total_seconds() / 3600
    first_24h_tr = df_t_time[df_t_time['hours_since_reg'] <= 24].groupby('id_user').size().reset_index(name='tr_count_24h')
    amount_3h = df_t_time[df_t_time['hours_since_reg'] <= 3].groupby('id_user')['amount'].sum().reset_index(name='amount_spent_3h')
    amount_24h = df_t_time[df_t_time['hours_since_reg'] <= 24].groupby('id_user')['amount'].sum().reset_index(name='amount_spent_24h')
    
    temp_time = pd.merge(df_t.groupby('id_user')['timestamp_tr'].min().reset_index(), 
                         df_u[['id_user', 'timestamp_reg']], on='id_user', how='left')
    temp_time['seconds_to_first_tr'] = (temp_time['timestamp_tr'] - temp_time['timestamp_reg']).dt.total_seconds()
    temp_time['first_tr_hour'] = temp_time['timestamp_tr'].dt.hour
    temp_time['first_tr_dayofweek'] = temp_time['timestamp_tr'].dt.dayofweek
    time_feat = temp_time[['id_user', 'seconds_to_first_tr', 'first_tr_hour', 'first_tr_dayofweek']]
    
    # 4. RATIO ТА 🔥 FAIL STREAK
    df_t['is_fail'] = (df_t['status'] == 'fail').astype(int)
    df_t['fail_block'] = (df_t['is_fail'] != df_t.groupby('id_user')['is_fail'].shift()).cumsum()
    df_t['streak'] = df_t.groupby(['id_user', 'fail_block']).cumcount() + 1
    df_t['fail_streak_calc'] = df_t['streak'] * df_t['is_fail']
    fail_streak_feat = df_t.groupby('id_user')['fail_streak_calc'].max().reset_index(name='max_fail_streak')

    ratio_feat = df_t.groupby('id_user').agg(
        success_c=('status', lambda x: (x == 'success').sum()),
        fail_c=('status', lambda x: (x == 'fail').sum())
    ).reset_index()
    ratio_feat['fail_to_success_ratio'] = ratio_feat['fail_c'] / (ratio_feat['success_c'] + 1)
    ratio_feat.drop(columns=['success_c', 'fail_c'], inplace=True)
    
    # 5. ФІНАНСИ ТА КОПІЙКИ
    df_t['amount_cents'] = (df_t['amount'] - np.floor(df_t['amount'])).round(2)
    fin_feat = df_t.groupby('id_user').agg(
        total_amount=('amount', 'sum'),
        max_amount=('amount', 'max'),
        avg_amount=('amount', 'mean'),
        std_amount=('amount', 'std'),
        avg_cents=('amount_cents', 'mean'),
        zero_cents_count=('amount_cents', lambda x: (x == 0.00).sum()),
        ninetynine_cents_count=('amount_cents', lambda x: (x == 0.99).sum())
    ).reset_index()
    fin_feat['std_amount'] = fin_feat['std_amount'].fillna(0)
    
    # 6. ГЕОГРАФІЯ ТА ВАЛЮТА (Currency)
    geo_feat = df_t.groupby('id_user').agg(
        top_card_country=('card_country', lambda x: x.mode()[0] if not x.mode().empty else 'unknown'),
        top_payment_country=('payment_country', lambda x: x.mode()[0] if not x.mode().empty else 'unknown'),
        top_currency=('currency', lambda x: x.mode()[0] if not x.mode().empty else 'unknown'),
        unique_card_countries=('card_country', 'nunique'),
        unique_payment_countries=('payment_country', 'nunique'),
        unique_currencies=('currency', 'nunique')
    ).reset_index()
    
    geo_feat = pd.merge(geo_feat, df_u[['id_user', 'reg_country']], on='id_user', how='left')
    geo_feat['total_geo_mismatches'] = (
        (geo_feat['reg_country'] != geo_feat['top_card_country']).astype(int) +
        (geo_feat['reg_country'] != geo_feat['top_payment_country']).astype(int) +
        (geo_feat['top_card_country'] != geo_feat['top_payment_country']).astype(int)
    )
    geo_feat.drop(columns=['reg_country'], inplace=True)
    
    card_feat = df_t.groupby('id_user').agg(
        unique_card_holders=('card_holder', 'nunique'),
        unique_card_types=('card_type', 'nunique'), 
        unique_card_brands=('card_brand', 'nunique')
    ).reset_index()
    
    # ГЕНДЕРНИЙ КОНФЛІКТ
    try:
        import gender_guesser.detector as gender
        d = gender.Detector()
        top_card_holder = df_t.groupby('id_user')['card_holder'].agg(lambda x: x.mode()[0] if not x.mode().empty else 'unknown').reset_index(name='top_card_holder')
        top_card_holder['top_first_name'] = top_card_holder['top_card_holder'].str.split().str[0].str.capitalize()
        unique_names = top_card_holder['top_first_name'].unique()
        gender_dict = {name: d.get_gender(name) for name in unique_names}
        top_card_holder['card_gender_pred'] = top_card_holder['top_first_name'].map(gender_dict)
        top_card_holder = pd.merge(top_card_holder, df_u[['id_user', 'gender']], on='id_user', how='left')
        
        def check_gender_mismatch(row):
            u_gen, c_gen = str(row['gender']).lower(), str(row['card_gender_pred']).lower()
            if u_gen == 'unknown' or c_gen == 'unknown' or c_gen == 'andy': return 0 
            if (u_gen == 'male' and 'female' in c_gen) or (u_gen == 'female' and 'male' in c_gen): return 1 
            return 0
            
        top_card_holder['gender_mismatch'] = top_card_holder.apply(check_gender_mismatch, axis=1)
        gender_mismatch_feat = top_card_holder[['id_user', 'gender_mismatch']]
    except ImportError:
        gender_mismatch_feat = pd.DataFrame({'id_user': df_u['id_user'], 'gender_mismatch': 0})

    # ПІВОТИ ПОМИЛОК ТА ТИПІВ ТРАНЗАКЦІЙ
    error_pivot = pd.crosstab(df_t['id_user'], df_t['error_group']).fillna(0)
    error_pivot.columns = [f"err_{col}" for col in error_pivot.columns]
    error_pivot = error_pivot.reset_index()
    
    type_pivot = pd.crosstab(df_t['id_user'], df_t['transaction_type']).fillna(0)
    type_pivot.columns = [f"type_{col}" for col in type_pivot.columns]
    type_pivot = type_pivot.reset_index()
    
    # 7. ПОШТА, FUZZY MATCHING ТА АНАЛІЗ БОТ-ПАТТЕРНІВ
    df_u[['email_id', 'email_domain']] = df_u['email'].str.extract(r'([^@]+)@(.+)')
    df_u['email_domain'] = df_u['email_domain'].fillna('unknown')
    df_u['email_tld'] = df_u['email_domain'].apply(lambda x: str(x).split('.')[-1] if '.' in str(x) else 'unknown')
    rare_domains = df_u['email_domain'].value_counts()[df_u['email_domain'].value_counts() < 10].index
    df_u['email_domain_clean'] = df_u['email_domain'].apply(lambda x: 'other' if x in rare_domains else x)
    
    # Аналіз цифр у пошті
    df_u['email_length'] = df_u['email_id'].astype(str).apply(len)
    df_u['email_digits_count'] = df_u['email_id'].astype(str).apply(lambda x: sum(c.isdigit() for c in x))
    df_u['email_digits_ratio'] = df_u['email_digits_count'] / (df_u['email_length'] + 0.001)

    top_holder_df = df_t.groupby('id_user')['card_holder'].agg(lambda x: x.mode()[0] if not x.mode().empty else 'unknown').reset_index(name='top_card_holder')
    similarity_df = pd.merge(top_holder_df, df_u[['id_user', 'email_id']], on='id_user', how='left')
    
    def string_similarity(row):
        name = str(row['top_card_holder']).lower().replace(' ', '')
        email_prefix = str(row['email_id']).lower()
        if name == 'unknown' or email_prefix == 'unknown' or email_prefix == 'nan': 
            return 0.5 
        try:
            return difflib.SequenceMatcher(None, name, email_prefix).ratio()
        except Exception:
            return 0.5
        
    similarity_df['name_email_similarity'] = similarity_df.apply(string_similarity, axis=1)
    similarity_feat = similarity_df[['id_user', 'name_email_similarity']]

    # 8. ROLLING WINDOWS 
    df_t['rolling_mean_3'] = df_t.groupby('id_user')['amount'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df_t['amount_vs_rolling3'] = df_t['amount'] / (df_t['rolling_mean_3'] + 0.001)
    rolling_feat = df_t.groupby('id_user').agg(
        max_amount_vs_rolling3=('amount_vs_rolling3', 'max'),
        mean_amount_vs_rolling3=('amount_vs_rolling3', 'mean')
    ).reset_index()

    # 9. ОБ'ЄДНАННЯ
    df_final = df_u.copy()
    merge_list = [n_transactions, suspicious_err, network_feat, cards_per_user, 
                  velocity_feat, first_24h_tr, amount_3h, amount_24h, time_feat, ratio_feat, fail_streak_feat, fin_feat, 
                  geo_feat, card_feat, gender_mismatch_feat, error_pivot, type_pivot, similarity_feat, rolling_feat]
    
    for feat_df in merge_list:
        df_final = pd.merge(df_final, feat_df, on='id_user', how='left')
        
    df_final.fillna({
        'n_transactions': 0, 'suspicious_errors_count': 0, 'max_users_per_card': 1, 
        'unique_cards_tried': 0, 'avg_time_between_tr': -1, 'min_time_between_tr': -1, 
        'tr_count_24h': 0, 'seconds_to_first_tr': -1, 'first_tr_hour': -1, 'first_tr_dayofweek': -1,
        'amount_spent_3h': 0, 'amount_spent_24h': 0, 'max_fail_streak': 0,
        'fail_to_success_ratio': 0, 'total_amount': 0, 'max_amount': 0, 'avg_amount': 0, 
        'std_amount': 0, 'avg_cents': 0, 'zero_cents_count': 0, 'ninetynine_cents_count': 0,
        'unique_card_countries': 0, 'unique_payment_countries': 0, 'unique_currencies': 0,
        'total_geo_mismatches': 0, 'unique_card_holders': 0, 'unique_card_types': 0, 
        'unique_card_brands': 0, 'gender_mismatch': 0, 
        'top_card_country': 'unknown', 'top_payment_country': 'unknown', 'top_currency': 'unknown',
        'top_card_brand': 'unknown', 'top_card_type': 'unknown',
        'email_length': 0, 'email_digits_count': 0, 'email_digits_ratio': 0,
        'name_email_similarity': 0.5,
        'max_amount_vs_rolling3': 1.0, 'mean_amount_vs_rolling3': 1.0 
    }, inplace=True)
    
    for col in error_pivot.columns.tolist() + type_pivot.columns.tolist():
        if col != 'id_user': df_final[col] = df_final.get(col, 0).fillna(0)
            
    # 10. FEATURE CROSSES 
    df_final['ratio_x_cards'] = df_final['fail_to_success_ratio'] * df_final['unique_cards_tried']
    df_final['ratio_x_holders'] = df_final['fail_to_success_ratio'] * df_final['unique_card_holders']
    df_final['mismatch_x_cards'] = df_final['total_geo_mismatches'] * df_final['unique_cards_tried']
    df_final['cross_country_currency'] = df_final['top_payment_country'].astype(str) + "_" + df_final['top_currency'].astype(str)
    
    if 'err_antifraud' in df_final.columns:
        df_final['ratio_x_antifraud'] = df_final['fail_to_success_ratio'] * df_final['err_antifraud']
        
    df_final['velocity_3h_ratio'] = df_final['amount_spent_3h'] / (df_final['total_amount'] + 0.001)
    df_final['velocity_24h_ratio'] = df_final['amount_spent_24h'] / (df_final['total_amount'] + 0.001)
    df_final['zero_cents_ratio'] = df_final['zero_cents_count'] / (df_final['n_transactions'] + 0.001)
    
    return df_final

print("\n⏳ Генеруємо фічі для Train...")
df_train_final = create_features(users, transactions)
print(f"✅ Готово! Згенеровано {df_train_final.shape[1]} колонок.")

In [ ]:
# 3.5. Стиснення пам'яті

In [ ]:
def reduce_mem_usage(df):
    print("🗜️ Починаю стиснення пам'яті...")
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        # 🔥 ВИПРАВЛЕНО: Ігноруємо тип datetime, щоб уникнути помилки '>' not supported
        if col_type != object and str(col_type) != 'category' and 'datetime' not in str(col_type) and col != 'is_fraud':
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max: df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max: df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024**2
    print(f"📉 Пам'ять зменшено з {start_mem:.2f} MB до {end_mem:.2f} MB!")
    return df

df_train_final = reduce_mem_usage(df_train_final)

# 4. ПІДГОТОВКА X ТА y

In [ ]:
cols_to_drop = ['is_fraud', 'id_user', 'timestamp_reg', 'email', 'email_id', 'email_domain', 'max_users_per_card']

X = df_train_final.drop(columns=[col for col in cols_to_drop if col in df_train_final.columns], errors='ignore')
y = df_train_final['is_fraud']

# 🔥 ЗБЕРІГАЄМО КАТЕГОРІЇ ДЛЯ МАЙБУТНЬОГО ТЕСТОВОГО ДАТАСЕТУ
category_mapping = {}
cat_features = X.select_dtypes(include=['object', 'string']).columns.tolist()

for col in cat_features:
    X[col] = X[col].fillna('unknown').astype(str)
    known_categories = list(set(X[col].unique()) | {'unknown'})
    
    # Зберігаємо у словник
    category_mapping[col] = known_categories
    
    # Переводимо у формат Category
    X[col] = pd.Categorical(X[col], categories=known_categories)

print(f"Дані готові! Категоріальні фічі збережено в словник та переведено у формат Category: {cat_features}")

# 5. OPTUNA - ПОШУК ІДЕАЛУ (ОПЦІОНАЛЬНО)

In [ ]:
'''
# ==========================================
# [КЛІТИНКА 5: OPTUNA ДЛЯ LIGHTGBM ТА XGBOOST]
# ==========================================
import optuna
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("⚙️ Перетворюємо категоріальні фічі для LGBM та XGBoost...")
# XGBoost та LightGBM вимагають тип 'category' замість 'object'/'string'
X_opt = X.copy()
for col in cat_features:
    X_opt[col] = X_opt[col].astype('category')

# Фіксуємо K-Fold
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42) # 3 фолди для швидкості пошуку

# ==========================================
# 1. OPTUNA ДЛЯ LIGHTGBM
# ==========================================
def objective_lgb(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss', # 🔥 ВИПРАВЛЕНО: тепер early_stopping знає, куди дивитися
        'boosting_type': 'gbdt',
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 5.0, 12.0),
        'random_state': 42,
        'verbose': -1,
        'n_jobs': -1 # Використовувати всі ядра процесора
    }

    oof_preds = np.zeros(len(X_opt))
    for train_idx, val_idx in skf.split(X_opt, y):
        X_tr, y_tr = X_opt.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X_opt.iloc[val_idx], y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr, 
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
        )
        oof_preds[val_idx] = model.predict_proba(X_va)[:, 1]

    # Шукаємо найкращий поріг локально для оцінки
    best_f1 = 0
    for th in np.arange(0.3, 0.9, 0.05):
        f1 = f1_score(y, (oof_preds >= th).astype(int), average='macro')
        if f1 > best_f1: best_f1 = f1
    return best_f1

# ==========================================
# 2. OPTUNA ДЛЯ XGBOOST (З ПІДТРИМКОЮ GPU)
# ==========================================
def objective_xgb(trial):
    params = {
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 5.0, 12.0),
        'tree_method': 'hist',
        'device': 'cuda', # 🔥 NVIDIA RTX 5070 ON!
        'enable_categorical': True, # 🔥 Магія XGBoost для категорій
        'random_state': 42,
        'eval_metric': 'logloss',
        'early_stopping_rounds': 100
    }

    oof_preds = np.zeros(len(X_opt))
    for train_idx, val_idx in skf.split(X_opt, y):
        X_tr, y_tr = X_opt.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X_opt.iloc[val_idx], y.iloc[val_idx]
        
        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        oof_preds[val_idx] = model.predict_proba(X_va)[:, 1]

    best_f1 = 0
    for th in np.arange(0.3, 0.9, 0.05):
        f1 = f1_score(y, (oof_preds >= th).astype(int), average='macro')
        if f1 > best_f1: best_f1 = f1
    return best_f1

print("\n🚀 Починаємо полювання на параметри LightGBM...")
study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=15) # 15 спроб достатньо

print("\n🚀 Починаємо полювання на параметри XGBoost...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=15)

print("\n======================================")
print("🏆 НАЙКРАЩІ ПАРАМЕТРИ ЗНАЙДЕНО!")
print("======================================")
print("🔥 LightGBM Best Params:", study_lgb.best_params)
print("🔥 XGBoost Best Params:", study_xgb.best_params)
'''

# 6. ТРЕНУВАННЯ ГЕТЕРОГЕННОГО АНСАМБЛЮ

In [ ]:
import lightgbm as lgb # 🔥 ДОДАНО СЮДИ, ЩОБ УНИКНУТИ ПОМИЛКИ NameError, якщо перша клітинка не була запущена

print("\n🚀 Запускаємо 4-x голового Дракона (CatBoost + LightGBM + XGBoost + CatBoost Focal)...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# OOF прогнози (для оцінки та блендінгу)
oof_cat = np.zeros(len(X)) 
oof_cat_focal = np.zeros(len(X))
oof_lgb = np.zeros(len(X)) 
oof_xgb = np.zeros(len(X)) 

# Списки для збереження натренованих моделей (Production Ready!)
models_cat = []
models_cat_focal = []
models_lgb = []
models_xgb = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Тренуємо Fold {fold + 1} / 5 ---")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    print("  🏃‍♂️ 1. CatBoost (Standart)...", end="")
    model_cat = CatBoostClassifier(
        iterations=2500, learning_rate=0.0354, depth=6, scale_pos_weight=8.55, 
        l2_leaf_reg=7.45, eval_metric='F1', cat_features=cat_features, 
        early_stopping_rounds=200, random_seed=42 + fold, verbose=0, task_type='GPU', devices='0'
    )
    model_cat.fit(X_train, y_train, eval_set=(X_val, y_val))
    oof_cat[val_idx] = model_cat.predict_proba(X_val)[:, 1]
    models_cat.append(model_cat) # ЗБЕРІГАЄМО
    print(f" Готово")
    
    print("  🎯 2. CatBoost (Focal Loss, CPU)...", end="")
    model_cat_focal = CatBoostClassifier(
        iterations=2000, learning_rate=0.04, depth=6,
        loss_function='Focal:focal_alpha=0.25;focal_gamma=2.0',
        eval_metric='F1', cat_features=cat_features, 
        early_stopping_rounds=150, random_seed=99 + fold, verbose=0, 
        task_type='CPU', thread_count=-1
    )
    model_cat_focal.fit(X_train, y_train, eval_set=(X_val, y_val))
    oof_cat_focal[val_idx] = model_cat_focal.predict_proba(X_val)[:, 1]
    models_cat_focal.append(model_cat_focal) # ЗБЕРІГАЄМО
    print(f" Готово")
    
    print("  🏃‍♂️ 3. LightGBM...", end="")
    model_lgb = LGBMClassifier(
        n_estimators=1500, learning_rate=0.0185, num_leaves=98, max_depth=8,
        min_child_samples=100, subsample=0.70, colsample_bytree=0.60,
        scale_pos_weight=5.20, random_state=42 + fold, n_jobs=-1, verbose=-1
    )
    # 🔥 ВИПРАВЛЕНО: Додано офіційний lgb.early_stopping, щоб модель не перенавчалася 
    model_lgb.fit(
        X_train, y_train, 
        eval_set=[(X_val, y_val)], 
        callbacks=[lgb.early_stopping(stopping_rounds=150, verbose=False)]
    )
    oof_lgb[val_idx] = model_lgb.predict_proba(X_val)[:, 1]
    models_lgb.append(model_lgb) # ЗБЕРІГАЄМО
    print(" Готово")
    
    print("  🏃‍♂️ 4. XGBoost...", end="")
    model_xgb = XGBClassifier(
        n_estimators=1500, learning_rate=0.0334, max_depth=9, min_child_weight=10,
        subsample=0.98, colsample_bytree=0.61, scale_pos_weight=11.11,
        enable_categorical=True, tree_method='hist', device='cuda',
        random_state=42 + fold, early_stopping_rounds=150, eval_metric='logloss'
    )
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
    models_xgb.append(model_xgb) # ЗБЕРІГАЄМО
    print(" Готово")

print("\n🎉 Усі 20 моделей успішно натреновано та збережено в пам'яті!")

In [ ]:
import optuna 

def blend_objective(trial):
    w1 = trial.suggest_float('w1', 0.0, 1.0) 
    w2 = trial.suggest_float('w2', 0.0, 1.0) 
    w3 = trial.suggest_float('w3', 0.0, 1.0) 
    w4 = trial.suggest_float('w4', 0.0, 1.0) 
    
    total = w1 + w2 + w3 + w4
    if total == 0: return 0
    w1, w2, w3, w4 = w1/total, w2/total, w3/total, w4/total
    
    blend = (oof_cat * w1) + (oof_cat_focal * w2) + (oof_lgb * w3) + (oof_xgb * w4)
    
    best_f1 = 0
    for th in np.arange(0.3, 0.9, 0.05):
        preds = (blend >= th).astype(int)
        f1 = f1_score(y, preds, average='macro')
        if f1 > best_f1: best_f1 = f1
    return best_f1

optuna.logging.set_verbosity(optuna.logging.WARNING)
blend_study = optuna.create_study(direction='maximize')
blend_study.optimize(blend_objective, n_trials=150)

best_w = blend_study.best_params
total_w = sum(best_w.values())
w_cat, w_focal, w_lgb, w_xgb = best_w['w1']/total_w, best_w['w2']/total_w, best_w['w3']/total_w, best_w['w4']/total_w

print(f"\n🥇 Ідеальні ваги знайдено!")
print(f"CatBoost: {w_cat:.3f} | Cat Focal: {w_focal:.3f} | LGBM: {w_lgb:.3f} | XGBoost: {w_xgb:.3f}")

# Блендінг для Train (OOF)
oof_probs_blend = (oof_cat * w_cat) + (oof_cat_focal * w_focal) + (oof_lgb * w_lgb) + (oof_xgb * w_xgb)

# 7. SMART POST-PROCESSING (КЛАСТЕРНІ ПОРОГИ)

In [ ]:
print("\n🔎 Шукаємо ГЛОБАЛЬНИЙ та кластерні пороги...")

# 1. Знаходимо Глобальний поріг (найбезпечніший запобіжник)
global_best_f1, global_best_th = 0, 0.5
for th in np.arange(0.1, 0.9, 0.01):
    preds = (oof_probs_blend >= th).astype(int)
    f1 = f1_score(y, preds, average='macro')
    if f1 > global_best_f1:
        global_best_f1 = f1
        global_best_th = th
        
print(f"🌍 ГЛОБАЛЬНИЙ Ідеальний поріг: {global_best_th:.2f} (Macro F1: {global_best_f1:.4f})")

# 2. Знаходимо Кластерні пороги (тільки для великих груп)
cluster_column = 'traffic_type' 
unique_clusters = X[cluster_column].unique()
best_thresholds = {}

y_pred_smart = np.zeros(len(y))

for cluster in unique_clusters:
    # Оптимізація на Train
    idx = (X[cluster_column] == cluster)
    y_cluster = y[idx]
    probs_cluster = oof_probs_blend[idx]
    
    # Захист від перенавчання: якщо кластер замалий, беремо глобальний поріг
    if idx.sum() < 200:
        best_thresholds[cluster] = global_best_th
        y_pred_smart[idx] = (probs_cluster >= global_best_th).astype(int)
        print(f"  📌 Кластер '{cluster}' (МАЛИЙ, N={idx.sum()}): Використовуємо Глобальний Поріг = {global_best_th:.2f}")
        continue
        
    best_f1, best_th = 0, global_best_th
    for th in np.arange(0.1, 0.9, 0.01):
        preds = (probs_cluster >= th).astype(int)
        f1 = f1_score(y_cluster, preds, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_th = th
            
    best_thresholds[cluster] = best_th
    y_pred_smart[idx] = (probs_cluster >= best_th).astype(int)
    print(f"  📌 Кластер '{cluster}' (N={idx.sum()}): Поріг = {best_th:.2f} | Local Macro F1 = {best_f1:.4f}")
    
final_macro = f1_score(y, y_pred_smart, average='macro')
final_binary = f1_score(y, y_pred_smart, average='binary')

print(f"======================================")
print(f"🏆 МАКСИМАЛЬНИЙ Macro F1 (Grandmaster Ensemble 4 Models): {final_macro:.4f}")
print(f"🔥 Binary F1: {final_binary:.4f}")
print(f"======================================")

print("\n=== ФІНАЛЬНИЙ ЗВІТ МОДЕЛІ (OOF) ===")
print(classification_report(y, y_pred_smart))

# 8. PRODUCTION INFERENCE (ЗАВАНТАЖЕННЯ TEST ТА SUBMISSION)

In [ ]:
print("\n📥 ЗАВАНТАЖЕННЯ ТА ОБРОБКА TEST (Production Mode)...")
test_users = pd.read_csv('test_users.csv')
test_transactions = pd.read_csv('test_transactions.csv')

test_users.drop_duplicates(inplace=True)
test_transactions.drop_duplicates(inplace=True)
test_users.fillna('unknown', inplace=True)
test_transactions.fillna('none', inplace=True)
test_users["timestamp_reg"] = pd.to_datetime(test_users["timestamp_reg"], format='mixed', utc=True)
test_transactions["timestamp_tr"] = pd.to_datetime(test_transactions["timestamp_tr"], format='mixed', utc=True)

df_test_final = create_features(test_users, test_transactions)
X_test = df_test_final.drop(columns=[col for col in cols_to_drop if col in df_test_final.columns], errors='ignore')

# Синхронізація колонок 
for col in X.columns:
    if col not in X_test.columns:
        X_test[col] = 0
X_test = X_test[X.columns]

# 🔥 ЗАСТОСОВУЄМО ЗБЕРЕЖЕНИЙ СЛОВНИК КАТЕГОРІЙ
print("⏳ Застосовуємо збережені категорії до Test...")
for col in cat_features:
    X_test[col] = X_test[col].fillna('unknown').astype(str)
    
    # Якщо прийшла нова країна/картка, якої не було в Train - замінюємо на 'unknown'
    known_set = set(category_mapping[col])
    X_test[col] = X_test[col].apply(lambda x: x if x in known_set else 'unknown')
    
    # Переводимо в Categorical використовуючи структуру з Train
    X_test[col] = pd.Categorical(X_test[col], categories=category_mapping[col])

# Отримуємо середні передбачення зі збережених моделей
print("🤖 Робимо передбачення ансамблем з 20 моделей...")
test_preds_cat = np.mean([m.predict_proba(X_test)[:, 1] for m in models_cat], axis=0)
test_preds_focal = np.mean([m.predict_proba(X_test)[:, 1] for m in models_cat_focal], axis=0)
test_preds_lgb = np.mean([m.predict_proba(X_test)[:, 1] for m in models_lgb], axis=0)
test_preds_xgb = np.mean([m.predict_proba(X_test)[:, 1] for m in models_xgb], axis=0)

# Блендінг
test_probs_blend = (test_preds_cat * w_cat) + (test_preds_focal * w_focal) + (test_preds_lgb * w_lgb) + (test_preds_xgb * w_xgb)

# Застосування порогів
# 🔥 ВИПРАВЛЕННЯ: Спочатку застосовуємо глобальний поріг для ВСІХ юзерів
# Це гарантує, що якщо в Test з'явиться абсолютно новий traffic_type, 
# він не отримає автоматично "0", а буде оцінений за загальним правилом.
test_pred_smart = (test_probs_blend >= global_best_th).astype(int)

for cluster in unique_clusters:
    idx_test = (X_test[cluster_column] == cluster)
    if idx_test.sum() > 0:
        th = best_thresholds.get(cluster, global_best_th) # 🔥 Використовуємо захищений Fallback
        test_pred_smart[idx_test] = (test_probs_blend[idx_test] >= th).astype(int)

# 🔥 ЗБЕРЕЖЕННЯ SUBMISSION.CSV
print("\n💾 Зберігаємо фінальний файл submission.csv...")
submission = pd.DataFrame({
    'id_user': df_test_final['id_user'],
    'is_fraud': test_pred_smart.astype(int)
})
submission.to_csv('submission.csv', index=False)
print("✅ submission.csv збережено! Готово до відправки на пошту.")

In [ ]:
# Перевірка кількості рядків
print("=== 📏 ПЕРЕВІРКА РОЗМІРУ ФАЙЛІВ ===")
print(f"Кількість записів у test_users: {len(test_users)}")
print(f"Кількість записів у submission: {len(submission)}")

if len(test_users) == len(submission):
    print("✅ Все супер! Кількість записів ідеально збігається.")
else:
    print(f"🚨 Увага! Різниця у {abs(len(test_users) - len(submission))} записів. Перевір drop_duplicates.")

In [ ]:
# Перевірка розподілу фроду (Sanity Check)

# 1. Рахуємо відсоток фроду в Train (історичні дані)
train_fraud_count = y.sum()
train_total = len(y)
train_fraud_rate = (train_fraud_count / train_total) * 100

# 2. Рахуємо відсоток фроду в Test (наші прогнози з submission)
test_fraud_count = submission['is_fraud'].sum()
test_total = len(submission)
test_fraud_rate = (test_fraud_count / test_total) * 100

# 3. Виводимо красиве порівняння
print("=== ⚖️ ПОРІВНЯННЯ РОЗПОДІЛУ ФРОДУ ===
print(f"📉 Train Data: {train_fraud_rate:.2f}% фроду ({train_fraud_count} шахраїв з {train_total} юзерів)")
print(f"📈 Test Data:  {test_fraud_rate:.2f}% фроду ({test_fraud_count} шахраїв з {test_total} юзерів)")

# 4. Аналіз результату
diff = abs(train_fraud_rate - test_fraud_rate)
print(f"\n📊 Різниця: {diff:.2f}%")

if diff < 1.0:
    print("✅ ІДЕАЛЬНО! Відсоток фроду в прогнозах майже ідентичний історичним даним. Модель чудово збалансована.")
elif diff < 3.0:
    print("⚠️ НОРМАЛЬНО. Є невелика різниця, але це в межах норми для нових даних.")
elif test_fraud_rate > train_fraud_rate:
    print("🚨 УВАГА! Модель розмітила занадто багато юзерів як фрод (False Positives). Можливо, поріг занизький.")
else:
    print("🚨 УВАГА! Модель зловила підозріло мало фроду. Можливо, поріг занадто високий.")

# 9. ADVERSARIAL VALIDATION (ДЕТЕКТОР ЗСУВУ ДАНИХ)

In [ ]:
# ==========================================
# [КЛІТИНКА 9: ADVERSARIAL VALIDATION (ДЕТЕКТОР ЗСУВУ ДАНИХ)]
# ==========================================
print("🕵️‍♂️ Запускаємо Adversarial Validation...")
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier

# 0. ПІДГОТОВКА X_test (Якщо він ще не був створений у попередніх клітинках)
if 'X_test' not in locals() and 'X_test' not in globals():
    print("⏳ Змінну 'X_test' не знайдено. Генерую тестові дані автоматично...")
    
    # --- ВИПРАВЛЕННЯ: Завантажуємо сирі тестові дані, якщо їх немає в пам'яті ---
    if 'test_users' not in locals() and 'test_users' not in globals():
        print("   📥 Завантажую test_users.csv...")
        test_users = pd.read_csv('test_users.csv')
        test_users["timestamp_reg"] = pd.to_datetime(test_users["timestamp_reg"], format='mixed', utc=True)
        test_users.drop_duplicates(inplace=True)
        test_users.fillna('unknown', inplace=True)
        
    if 'test_transactions' not in locals() and 'test_transactions' not in globals():
        print("   📥 Завантажую test_transactions.csv...")
        test_transactions = pd.read_csv('test_transactions.csv')
        test_transactions["timestamp_tr"] = pd.to_datetime(test_transactions["timestamp_tr"], format='mixed', utc=True)
        test_transactions.drop_duplicates(inplace=True)
        test_transactions.fillna('none', inplace=True)
    # -------------------------------------------------------------------------
    
    test_df_final = create_features(test_users, test_transactions)
    
    cols_to_drop = ['is_fraud', 'id_user', 'timestamp_reg', 'email', 'email_id', 'email_domain', 'max_users_per_card']
    X_test = test_df_final.drop(columns=[col for col in cols_to_drop if col in test_df_final.columns], errors='ignore')

    # Синхронізація колонок з X
    for col in X.columns:
        if col not in X_test.columns: 
            X_test[col] = 0
            
    # Жорстко фіксуємо порядок
    X_test = X_test[X.columns]

    # Категоріальні змінні - в текст
    cat_features = X.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    for col in cat_features:
        X_test[col] = X_test[col].fillna('unknown').astype(str)
        if hasattr(X[col], 'cat'): # Якщо в X це категорія (для XGBoost/LGBM)
            X_test[col] = X_test[col].astype('category')
            
    print("✅ X_test успішно згенеровано!")

# 1. Перевіряємо, чи всі колонки збігаються
assert list(X.columns) == list(X_test.columns), "Колонки в X та X_test повинні бути ідентичними!"

# 2. Створюємо нові мітки: 0 для Train, 1 для Test
y_adv_train = np.zeros(len(X))
y_adv_test = np.ones(len(X_test))

# 3. Зливаємо дані в один великий датасет
X_adv = pd.concat([X, X_test], ignore_index=True)
y_adv = np.concatenate([y_adv_train, y_adv_test])

# Переконуємось, що категоріальні змінні мають правильний тип
cat_features = X.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
for col in cat_features:
    X_adv[col] = X_adv[col].astype(str)

# 4. Розбиваємо на трейн і валідацію для Adversarial моделі
X_adv_train, X_adv_val, y_adv_train_split, y_adv_val_split = train_test_split(
    X_adv, y_adv, test_size=0.3, random_state=42, stratify=y_adv
)

# 5. Тренуємо "Модель-Детектив"
print("⏳ Тренуємо CatBoost для пошуку розбіжностей між Train та Test...")
adv_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=5,
    eval_metric='AUC',
    cat_features=cat_features,
    random_seed=42,
    verbose=100,
    task_type='GPU', # Використовуємо вашу RTX 5070 для швидкості
    devices='0'
)

adv_model.fit(X_adv_train, y_adv_train_split, eval_set=(X_adv_val, y_adv_val_split))

# 6. Оцінюємо результат
adv_preds = adv_model.predict_proba(X_adv_val)[:, 1]
adv_auc = roc_auc_score(y_adv_val_split, adv_preds)

print(f"\n======================================")
print(f"🎯 ADVERSARIAL ROC-AUC SCORE: {adv_auc:.4f}")
print(f"======================================")

if adv_auc < 0.60:
    print("✅ СУПЕР! Ваші дані дуже схожі. Моделі нічого не загрожує (Data Drift відсутній).")
elif adv_auc < 0.75:
    print("⚠️ Є невеликий зсув даних. Треба перевірити ТОП-фічі нижче.")
else:
    print("🚨 ТРИВОГА! Дані сильно змінилися! Модель перенавчається на фічах із ТОПу!")

# 7. Аналіз фічей-"зрадників"
adv_importances = pd.DataFrame({
    'Feature': X_adv.columns, 
    'Importance': adv_model.get_feature_importance()
}).sort_values(by='Importance', ascending=False)

print("\n🔥 ТОП-10 ФІЧЕЙ, ЯКІ ВІДРІЗНЯЮТЬ TRAIN ВІД TEST (Можливі 'зрадники'):")
print(adv_importances.head(10))

# Візуалізація
plt.figure(figsize=(10, 6))
plt.barh(adv_importances['Feature'].head(15)[::-1], adv_importances['Importance'].head(15)[::-1], color='crimson')
plt.title(f'Adversarial Feature Importance (ROC-AUC: {adv_auc:.4f})')
plt.xlabel('Важливість для відрізнення Train/Test')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# 1. Відбираємо ТІЛЬКИ числові колонки (ігноруємо текст типу 'DEBIT')
X_numeric = X.select_dtypes(include=['int16', 'int32', 'int64', 'float16', 'float32', 'float64'])

# 2. Шукаємо "клонів" (фічі, які корелюють між собою на > 95%)
corr_matrix = X_numeric.corr().abs()

# 3. Відкидаємо нижній трикутник матриці, щоб не дублювати пари
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# 4. Знаходимо колонки на видалення
to_drop_clones = [column for column in upper.columns if any(upper[column] > 0.95)]

print(f"🚨 Знайдено числових фічей-клонів: {len(to_drop_clones)}")
print("Список на видалення:", to_drop_clones)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("🔎 Аналізуємо важливість фічей (Feature Importances)...")

# Отримуємо назви всіх колонок
features = X.columns

# Усереднюємо важливість фічей з усіх 5 фолдів моделі CatBoost (Standart)
# Можна також взяти models_lgb (model.feature_importances_)
importances = np.zeros(len(features))
for model in models_cat:
    importances += model.get_feature_importance() / len(models_cat)

# Створюємо зручну таблицю
df_importances = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\n🔥 ТОП-15 НАЙВАЖЛИВІШИХ ФІЧЕЙ:")
print(df_importances.head(15))

# Візуалізуємо результати для звіту
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=df_importances.head(15), palette='magma')
plt.title('ТОП-15 ключових фічей (CatBoost Ensemble)', fontsize=14, fontweight='bold')
plt.xlabel('Важливість (Feature Importance)', fontsize=12)
plt.ylabel('Фіча', fontsize=12)
plt.tight_layout()
plt.show()